# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayaahmed571/Flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The playbook ranks content for review based on observed content signals and the validated baseline/model outputs.

### Action 1 — Review for refresh
**Reason code:** `STALE_WITH_VISIBILITY`

Content is relatively old and still has meaningful historical visibility. These pages are prioritized for human review because refreshing them may be worth investigating.

### Action 2 — Lower priority
**Reason code:** `STALE_LOW_VISIBILITY`

Content is relatively old but has lower historical visibility. These pages are kept in the queue but receive lower priority than stale content with stronger visibility.

### Action 3 — No immediate action
Content that does not meet the conditions above is not prioritized by this baseline playbook.

The ranking is intended for decision-support and human review, not automatic publishing or content changes.

## 2. Intended use and limits

### Intended use

This playbook is intended to support content teams in prioritizing pages for human review.

The ranked actions can help reviewers decide which content may deserve attention first, based on observed historical performance and content freshness signals.

The output is decision-support, not an automatic content-management system.

### Limits

The playbook does not prove that refreshing a page will improve its performance.

The model and baseline were evaluated on the available dataset and under a grouped-by-client split. The measured precision provides directional evidence, but performance may differ for new clients or future data.

The recommendations should therefore be treated as prioritization suggestions that require human review.

## 3. Human review + the no-go list

### Human review rules

Every recommended action should be reviewed by a human before any content change is made.

The reviewer should check:

- Whether the page is still relevant to the target topic.
- Whether the observed performance decline is meaningful in context.
- Whether the page has strategic or business value.
- Whether the suggested refresh is appropriate for the content type.
- Whether there is a recent update or external reason that explains the observed signal.

The score and reason code should be treated as prioritization signals, not as final decisions.

### No-go list

The system should NOT automatically:

- Rewrite or publish content.
- Delete or redirect pages.
- Change SEO metadata without review.
- Decide that a page has no business value.
- Treat correlation or model predictions as proof of causation.
- Apply a refresh recommendation without human review.

## 4. Monitoring / retrain triggers

The playbook should be monitored over time to check whether its recommendations remain useful.

### Monitoring

The following should be checked periodically:

- Precision of the model on newly observed data.
- The distribution of recommended actions.
- The distribution of the main input features.
- The number of pages receiving each reason code.
- Whether human reviewers frequently reject the recommendations.

### Retrain / review triggers

The model or rule should be reviewed and potentially retrained when:

- Measured precision drops meaningfully compared with the validated result.
- The distribution of important features changes substantially.
- The data pipeline or feature definitions change.
- The content population changes substantially.
- Human reviewers consistently disagree with the recommendations.

These are monitoring and review triggers rather than automatic retraining rules.

## 5. Exports for the paper

The ranked action queue is exported for reuse in the research paper.

The CSV contains the ranked content actions, reason codes, and supporting signals used for prioritization.

The export is generated by the notebook so that the queue can be reproduced when the notebook is run again.

In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df.shape)

df["baseline_score"] = (
    (df["days_since_last_update"] >= 180).astype(int) * 2
    + (df["impressions_90d"] >= 500).astype(int)
)

df["reason_code"] = np.where(
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500),
    "STALE_WITH_VISIBILITY",
    np.where(
        df["days_since_last_update"] >= 180,
        "STALE_LOW_VISIBILITY",
        "NO_ACTION"
    )
)

df["action"] = np.where(
    df["reason_code"] == "STALE_WITH_VISIBILITY",
    "Review for refresh",
    np.where(
        df["reason_code"] == "STALE_LOW_VISIBILITY",
        "Lower priority",
        "No immediate action"
    )
)

print(df[
    ["content_id", "baseline_score", "reason_code", "action"]
].head(10))

(30000, 45)
             content_id  baseline_score reason_code               action
0  content_304f48230142               1   NO_ACTION  No immediate action
1  content_a1fb4e703a9e               1   NO_ACTION  No immediate action
2  content_9aa793d4d895               1   NO_ACTION  No immediate action
3  content_331d6c4de07b               1   NO_ACTION  No immediate action
4  content_d99b7a2d90ca               1   NO_ACTION  No immediate action
5  content_d4084a4bc775               1   NO_ACTION  No immediate action
6  content_9a34b442b552               0   NO_ACTION  No immediate action
7  content_a63219c6e95a               1   NO_ACTION  No immediate action
8  content_5e6c160719bc               1   NO_ACTION  No immediate action
9  content_c27558df2b0c               1   NO_ACTION  No immediate action


In [19]:
queue = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

queue = queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
]

print(queue.head(20))
print("Queue rows:", len(queue))

                 content_id  baseline_score            reason_code  \
16751  content_cf56e2e2e282               3  STALE_WITH_VISIBILITY   
16514  content_7368877ea310               3  STALE_WITH_VISIBILITY   
7021   content_1bfaa38ff26c               3  STALE_WITH_VISIBILITY   
21268  content_0a91db491d14               3  STALE_WITH_VISIBILITY   
11489  content_5feee3994adb               3  STALE_WITH_VISIBILITY   
12045  content_c2d929d83eaa               3  STALE_WITH_VISIBILITY   
698    content_b16bd7307b39               3  STALE_WITH_VISIBILITY   
5327   content_fe16a55cd13d               3  STALE_WITH_VISIBILITY   
26810  content_ecb6215e79fd               3  STALE_WITH_VISIBILITY   
20837  content_928af3e22c80               3  STALE_WITH_VISIBILITY   
22872  content_e3ff1b093148               3  STALE_WITH_VISIBILITY   
23215  content_bdbec75c1148               3  STALE_WITH_VISIBILITY   
26840  content_7f116ae1f6f5               3  STALE_WITH_VISIBILITY   
26799  content_77d4d

In [20]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(queue))
print("File exists:", os.path.exists(output_path))

Saved: work/outputs/baseline_action_score.csv
Rows: 30000
File exists: True


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.